# Goal 1: protein sequences to an alignment network

This notebook demonstrates the first project pipeline:

1. configure Biopython for local protein alignment;
2. define or load protein sequences;
3. compute every pairwise alignment score;
4. retain scores above an explicit threshold as graph edges.

Biopython performs the alignment itself. The project code only coordinates the pairwise calculations and graph construction.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from Bio import Align
from Bio.Align import substitution_matrices
from protein_alignment_networks import (
    pairwise_score_matrix,
    score_matrix_to_graph,
)

## 1. Configure Biopython

`mode='local'` requests the best matching subsequences. BLOSUM62 provides protein substitution scores. A gap costs -11 to open and -1 for each additional residue. These are starting parameters and must be recorded and tested in later experiments.

In [2]:
aligner = Align.PairwiseAligner(
    mode='local',
    substitution_matrix=substitution_matrices.load('BLOSUM62'),
    open_gap_score=-11,
    extend_gap_score=-1,
)
print(aligner.algorithm)

Gotoh local alignment algorithm


## 2. Inspect one alignment

For a single pair, Biopython can return both the numerical score and the aligned residues.

In [3]:
sequence_a = 'PAWHEAE'
sequence_b = 'HEAGAWGHEE'
alignments = aligner.align(sequence_a, sequence_b)
print(f'Score: {alignments.score:g}')
print(alignments[0] if len(alignments) else 'No positive local alignment')

Score: 17
target            1 AW-HE 5
                  0 ||-|| 5
query             4 AWGHE 9



## 3. Compute the pairwise score matrix

The project pipeline calls Biopython's `aligner.score()` for every unique pair and labels the resulting symmetric matrix.

In [4]:
sequences = {
    'protein_a': 'PAWHEAE',
    'protein_b': 'HEAGAWGHEE',
    'protein_c': 'MKTAYIAKQRQISFVKSHFSRQ',
    'protein_d': 'PAWHDQE',
}
scores = pairwise_score_matrix(sequences, aligner=aligner)
scores

protein_id,protein_a,protein_b,protein_c,protein_d
protein_id,,,,
protein_a,44.0,17.0,8.0,36.0
protein_b,17.0,62.0,8.0,19.0
protein_c,8.0,8.0,109.0,8.0
protein_d,36.0,19.0,8.0,46.0


## 4. Threshold the matrix into a graph

The threshold below only demonstrates the programming step. Raw alignment scores depend on protein length and composition, so a scientific graph will require normalisation, coverage checks, significance calibration, and threshold-sensitivity analysis.

In [5]:
demonstration_threshold = 20.0
graph = score_matrix_to_graph(scores, threshold=demonstration_threshold)
print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')
list(graph.edges(data=True))

Nodes: 4
Edges: 1


[('protein_a', 'protein_d', {'score': 36.0})]